# EvoHash OpenEvolve Run

Notebook entrypoint mirroring `Main_gigaevo_nb.ipynb`: configure one hash target, run checks, preview the generated problem (same prompt/seeds as GigaEvolve), launch evolution as a **background subprocess**, watch a non-drifting live log, then export the **N best programs** (top-N by score + the elite of every MAP-Elites cell, deduplicated).

Why a subprocess? OpenEvolve calls `asyncio.run()` and uses process-based parallel evaluation, which fight with a Jupyter kernel. Running `python -m evohash.stage3_openevolve.run` in its own process is the stable single entrypoint.

In [ ]:
import sys
from pathlib import Path

def find_project_root() -> Path:
    """Walk up from the notebook dir / cwd to the folder holding stage3_openevolve."""
    starts = []
    dh = globals().get("_dh")          # IPython: dirs the kernel has seen (startup dir first)
    if dh:
        starts += [Path(p) for p in dh]
    starts.append(Path.cwd())
    seen = set()
    for s in starts:
        s = s.resolve()
        for d in [s, *s.parents]:
            if d in seen:
                continue
            seen.add(d)
            if (d / "stage3_openevolve" / "notebook_api.py").is_file():
                return d
    raise RuntimeError(
        "Project root not found (no stage3_openevolve/notebook_api.py above "
        f"cwd={Path.cwd()}). Open the notebook from inside the repo."
    )

PROJECT_ROOT = find_project_root()
PACKAGE_PARENT = PROJECT_ROOT.parent
if str(PACKAGE_PARENT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PARENT))

for _m in [m for m in list(sys.modules) if m == "evohash" or m.startswith("evohash.")]:
    del sys.modules[_m]

try:
    from evohash.stage3_openevolve import notebook_api as evo
except ImportError as e:
    raise ImportError(
        f"{e}\n\n-> Partial sync? export.py / run.py / notebook_api.py must match "
        "(notebook_api imports extract_champions from export). Re-sync stage3_openevolve/."
    ) from e

from IPython.display import HTML, display
print("PROJECT_ROOT      =", PROJECT_ROOT)
print("notebook_api file =", evo.__file__)

## Config

In [ ]:
import os

# === Shared knobs (kept identical to the GigaEvolve notebook for parity) ===
HASH_ID = 'photodna'                 # phash | pdq | neuralhash | photodna
DATASET = str(PROJECT_ROOT / 'data' / 'dataset_v1_public.zip')
MAX_PAIRS = 30
BUDGET = 10000
SEED = 0
RESIZE_SIZE = 256
RUN_NAME = f'evohash_OpenEvolve_{HASH_ID}_EVO2'

# Initial / seed population (same pool GigaEvolve loads from initial_programs/)
SEED_CONFIG = str(PROJECT_ROOT / 'results' / 'attack_configs_optimized.json')
SEED_ATTACKS = 'all'              # 'all' or comma-separated attack ids

# Per-candidate evaluation parallelism (same evaluator both frameworks use)
EVAL_BACKEND = 'auto'             # auto | process | thread | sequential
WORKER_CAP = 5

# === Mutant budget (parity with GigaEvolve) ===
# GigaEvolve runs GENERATIONS * MAX_MUTATIONS_PER_GENERATION mutants. OpenEvolve
# instead takes one explicit ITERATIONS (each iteration = one LLM child program).
# Set ITERATIONS ~= GigaEvolve's GENERATIONS * MAX_MUTATIONS_PER_GENERATION.
ITERATIONS = 1000

# === OpenEvolve evolution hyperparameters (non-budget) ===
# Tuned for OUR regime: EXPENSIVE evaluation + few ITERATIONS + a strong, DIVERSE seed
# pool. OpenEvolve defaults assume thousands of iterations; we adapt.
#
# NUM_ISLANDS: independent sub-populations (best programs migrate every MIGRATION_INTERVAL).
#   ITERATIONS are SPLIT across islands -> with a small budget many islands means each
#   barely evolves and migration never fires. Use 1 for short runs; 2-3 only when
#   ITERATIONS >= ~100. MAP-Elites already gives us diversity, so islands are optional.
NUM_ISLANDS = 4
# POPULATION_SIZE: hard cap on stored programs (worst are pruned above it). At our scale
#   (seeds + ITERATIONS is small) we never want to prune, so keep it comfortably high.
POPULATION_SIZE = 500
# ARCHIVE_SIZE: the elite 'hall of fame' the exploitation path samples parents from.
#   Big enough to hold every distinct good strategy, small enough to stay focused.
ARCHIVE_SIZE = 100
#
# --- MAP-Elites grid (the diversity mechanism: keeps the best program PER CELL) ---
# FEATURE_DIMENSIONS: behaviour axes. asr is essential so a low-distortion/low-query
#   attack that FAILS sits in a different cell than a SUCCESSFUL one and can't evict it.
#     asr          = success rate (did the collision happen) -- the whole point
#     mean_l2_raw  = perceptual distortion (how visible the change is)
#     mean_queries = oracle query cost (how cheap the attack is)
#   Evaluator returns raw values; OpenEvolve min-max scales + bins each axis.
#   (Don't use 'score' as an axis -- it's the objective, not a behaviour.)
FEATURE_DIMENSIONS = ['asr', 'mean_l2_raw', 'mean_queries']
# FEATURE_BINS: resolution PER axis (dict). asr has few real levels (fail/partial/
#   success) -> keep it coarse; l2 & queries are continuous -> finer. With few programs,
#   too-fine bins put every program alone in a cell (no selection pressure); these keep
#   some in-cell competition while preserving distinct behaviours. (An int also works.)
FEATURE_BINS = {'asr': 4, 'mean_l2_raw': 5, 'mean_queries': 5}
# Migration only matters when NUM_ISLANDS > 1.
MIGRATION_INTERVAL = 10
MIGRATION_RATE = 0.2
#
# --- Parent selection per step (OpenEvolve names are counterintuitive) ---
#   EXPLORATION_RATIO  -> parent from the CURRENT ISLAND (diverse coverage)
#   EXPLOITATION_RATIO -> parent from the ELITE ARCHIVE (refine the winners)
#   leftover (1 - the two) -> a fully random program
# With a strong but partly-FAILING seed pool we bump exploration a touch so the diverse
# strategies (not only the single best) get mutation attempts within the small budget.
EXPLORATION_RATIO = 0.3
EXPLOITATION_RATIO = 0.6
# ELITE_SELECTION_RATIO is SEPARATE: fraction of the in-context INSPIRATION examples
#   taken from the top (vs diverse). Does NOT affect parent selection. (elite+expl+expt=1)
ELITE_SELECTION_RATIO = 0.1
# In-context examples shown to the LLM. More DIVERSE examples help it RECOMBINE
#   strategies (e.g. hybridise NES + SimBA); the cost is a longer prompt.
NUM_TOP_PROGRAMS = 3
NUM_DIVERSE_PROGRAMS = 3
# Full evaluation for every candidate -> fair vs GigaEvolve. Cascade pre-screen is OFF:
#   its gate compares combined_score (=-score, unbounded negative) so default thresholds
#   mis-gate, and a cheap 1-pair/low-budget screen can wrongly prune slow-but-good attacks.
CASCADE_EVALUATION = False
# Candidates evaluated concurrently inside OpenEvolve. 1 is safest on Windows;
# raise it if RAM allows (each worker loads dataset+model).
PARALLEL_EVALUATIONS = 1
RANDOM_SEED = 42
PREPOPULATE_DATABASE = True       # load the extra seeds into the population at iter 0
CHECKPOINT_INTERVAL = 100           # DB checkpoint cadence (clamped <= ITERATIONS below)


# === Evaluator ===
EVALUATOR_TIMEOUT     = 600   # сек на одну оценку кандидата (30 пар x budget); 300 часто мало
EVALUATOR_MAX_RETRIES = 1      # не пережигать время на повторных таймаутах

# === Export ===
TOP_N = 5                         # how many global top programs to save (plus per-cell elites)

# === LLM (same registry/aliases as GigaEvolve) ===
LLM_SOURCE = 'openrouter'         # openrouter | local
OPENROUTER_MODEL = 'qwen/qwen3.6-27b'  # paid/stable; use 'gpt_oss_120b_free' for :free
LOCAL_MODEL = 'qwen3.6-27b-fp8'          # used when LLM_SOURCE = 'local'
LOCAL_BASE_URL = 'http://127.0.0.1:8385/v1'
LLM_TEMPERATURE = 0.6
LLM_MAX_TOKENS = 40000
LLM_REQUEST_TIMEOUT = 600
MODEL = OPENROUTER_MODEL if LLM_SOURCE == 'openrouter' else LOCAL_MODEL

# === Flow switches ===
CHECK_LLM = True
REQUIRE_CHECKS_OK = True
RUN_EVOLUTION = True
MONITOR_IN_THIS_NOTEBOOK = True
EXPORT_AFTER_RUN = True

CHECKPOINT_INTERVAL = min(CHECKPOINT_INTERVAL, ITERATIONS)

# === Build the run spec (one source of truth for preview + subprocess) ===
SPEC = {
    'run_name': RUN_NAME,
    'output_dir': str(PROJECT_ROOT / 'outputs' / 'openevolve_runs' / RUN_NAME),
    'top_n': TOP_N,
    'dotenv': str(PROJECT_ROOT / '.env'),
    'evaluation': {
        'hash_id': HASH_ID,
        'dataset_zip': DATASET,
        'split': 'public',
        'max_pairs': MAX_PAIRS,
        'budget': BUDGET,
        'seed': SEED,
        'resize_size': RESIZE_SIZE,
        'parallel_backend': EVAL_BACKEND,
        'worker_cap': WORKER_CAP,
        'attack_config_path': SEED_CONFIG,
    },
    'llm': {
        'source': LLM_SOURCE,
        'model': MODEL,
        'base_url': LOCAL_BASE_URL if LLM_SOURCE == 'local' else '',
        'temperature': LLM_TEMPERATURE,
        'max_tokens': LLM_MAX_TOKENS,
        'timeout_s': LLM_REQUEST_TIMEOUT,
    },
    'problem': {
        'seed_config_path': SEED_CONFIG,
        'seed_attacks': [s.strip() for s in SEED_ATTACKS.split(',') if s.strip()],
        'iterations': ITERATIONS,
        'checkpoint_interval': CHECKPOINT_INTERVAL,
        'random_seed': RANDOM_SEED,
        'population_size': POPULATION_SIZE,
        'archive_size': ARCHIVE_SIZE,
        'num_islands': NUM_ISLANDS,
        'feature_dimensions': FEATURE_DIMENSIONS,
        'feature_bins': FEATURE_BINS,
        'migration_interval': MIGRATION_INTERVAL,
        'migration_rate': MIGRATION_RATE,
        'elite_selection_ratio': ELITE_SELECTION_RATIO,
        'exploration_ratio': EXPLORATION_RATIO,
        'exploitation_ratio': EXPLOITATION_RATIO,
        'num_top_programs': NUM_TOP_PROGRAMS,
        'num_diverse_programs': NUM_DIVERSE_PROGRAMS,
        'cascade_evaluation': CASCADE_EVALUATION,
        'parallel_evaluations': PARALLEL_EVALUATIONS,
        'prepopulate_database': PREPOPULATE_DATABASE,
        'evaluator_timeout_s': EVALUATOR_TIMEOUT,
        'evaluator_max_retries': EVALUATOR_MAX_RETRIES,
    },
}

display({
    'run_name': RUN_NAME,
    'hash': HASH_ID,
    'llm_source': LLM_SOURCE,
    'model': MODEL,
    'iterations': ITERATIONS,
    'pairs': MAX_PAIRS,
    'budget': BUDGET,
    'islands': NUM_ISLANDS,
    'population': POPULATION_SIZE,
    'feature_dimensions': FEATURE_DIMENSIONS,
    'parallel_evaluations': PARALLEL_EVALUATIONS,
    'top_n': TOP_N,
})

## Models

In [ ]:
display(evo.model_table())

## Checks

Non-throwing: if a required check fails, later cells skip the run.

In [ ]:
preflight = evo.preflight(
    dataset=DATASET,
    seed_config=SEED_CONFIG,
    llm=SPEC['llm'],          # ping LLM to catch dead key / empty-content (reasoning) models
    dotenv=SPEC['dotenv'],
    check_llm_connection=CHECK_LLM,
    require_openevolve=True,
)
display(evo.preflight_table(preflight))

CHECKS_OK = bool(preflight.get('ok')) or not REQUIRE_CHECKS_OK
color = '#16794c' if CHECKS_OK else '#b42318'
label = 'OK' if CHECKS_OK else 'BLOCKED'
display(HTML(f"<div style='font-weight:600;color:{color};padding:4px 0'>Checks: {label}</div>"))
if not CHECKS_OK:
    display(preflight)

In [ ]:
preview = evo.preview_problem(SPEC)
DRY_RUN_OK = bool(preview.get('problem_dir'))
print('problem_dir   :', preview.get('problem_dir'))
print('primary_attack:', preview.get('primary_attack'))
print('seed_attacks  :', preview.get('seed_attacks'))
print('db seeds      :', preview.get('db_seed_attacks'))
print('feature dims  :', preview.get('feature_dimensions'))
print('=' * 70)
print('SYSTEM PROMPT (sent to the LLM, identical content to GigaEvolve):')
print(preview.get('system_prompt'))
print('=' * 70)
print('config.yaml:')
print(preview.get('config_yaml'))

## Launch

Background mode runs `run.py` as a subprocess; all output is captured to `runner.log`.

In [ ]:
CAN_RUN = RUN_EVOLUTION and CHECKS_OK and DRY_RUN_OK

if not CAN_RUN:
    handle = {'ok': False, 'skipped': True, 'RUN_EVOLUTION': RUN_EVOLUTION,
              'CHECKS_OK': CHECKS_OK, 'DRY_RUN_OK': DRY_RUN_OK}
    display(handle)
else:
    handle = evo.start_openevolve_background(SPEC)
    display({k: v for k, v in handle.items() if k != 'proc'})
    print('output_dir:', handle['output_dir'])
    print('runner.log:', handle['runner_log'])

## Monitor

Non-drifting live log: a fixed-height `Textarea` whose value is replaced each refresh (ring-buffered), so the output never scrolls away. Blocks until the run finishes.

In [ ]:
if handle.get('proc') is not None and MONITOR_IN_THIS_NOTEBOOK:
    tail = evo.live_tail(handle, max_lines=500, height='420px', poll=1.0, follow=True)
    display(tail)
else:
    print('Monitor skipped (run not started or disabled). Status:', evo.run_status(handle) if handle.get('proc') else handle)

## Metrics

Best metrics per checkpoint (oldest -> newest).

In [ ]:
if handle.get('output_dir'):
    metrics_table = evo.read_checkpoint_metrics(handle['output_dir'])
    display(metrics_table)
    print('runner.log:', handle.get('runner_log'))
else:
    print('No output_dir on handle.')

## Export champions

Record-improvement chain: best seed → each program that **strictly beats** the running-best result
(equal metric = duplicate, skipped). Written to `<output_dir>/champions/` (one `.py` + `.json` per
champion) plus `aggregate.json` for comparison.

In [ ]:
from pathlib import Path

output_dir = evo.project_path("outputs/openevolve_runs") / RUN_NAME

handle = {
    "run_name": RUN_NAME,
    "output_dir": str(output_dir),
    "proc": None,
    "skipped": False,
}

agg = evo.extract_champions(handle["output_dir"])

if agg.get("ok"):
    print(
        "Champions:", agg["n_champions"],
        "| from", agg["n_programs_total"], "unique programs",
        "| records loaded:", agg.get("n_program_records_loaded"),
        "| checkpoints:", agg.get("n_checkpoints_scanned"),
        "| source:", agg.get("checkpoint_source"),
        "| total improvement (fitness):",
        round(agg.get("total_improvement_fitness", 0.0), 4),
    )
    print("Saved to:", str(Path(handle["output_dir"]) / "champions"), "(+ aggregate.json)")
    print("Coverage:", agg.get("coverage_note", ""))

    rows = [
        {k: c.get(k) for k in (
            "rank", "kind", "iteration_found", "generation", "id",
            "score", "asr", "mean_l2_raw", "mean_queries", "delta_fitness",
            "checkpoint_first_seen", "checkpoint_last_seen", "trace_path", "code_path",
        )}
        for c in agg.get("champions", [])
    ]

    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        display(rows)
else:
    print("extract_champions:", agg.get("reason"))